# YOLO12s + L-Ghost-inspired blocks + EMA-32
Adaptasi **LG + Ghost bottleneck** dari Chi et al. (2023), DOI: [10.1109/ACCESS.2023.3234108](https://doi.org/10.1109/ACCESS.2023.3234108).
Bukan replikasi utuh L-GhostNet: p-CA tidak digunakan; EMA-32 tetap di P3/P4.

Alur: clone branch → install repo → pretrained resmi YOLO12s → training dengan pruning → val → test → ZIP.
Aktifkan **Internet + satu GPU**. Jalankan dari session bersih. China dan dataset lokal dilatih **terpisah**.
Checkpoint default `best.pt` dipilih berdasarkan fitness validasi standar, bukan berdasarkan test atau mAP50 saja.
Tidak ada jaminan kenaikan mAP. README_LGHOST_EMA32.md menjelaskan metode dan batasannya.

In [ ]:
# 1. Clone dan install source yang benar, tanpa menimpa perubahan lokal.
import os
import subprocess
import sys
from pathlib import Path

if "ultralytics" in sys.modules:
    raise RuntimeError("Restart session Kaggle sebelum install agar tidak memakai modul dari repo lain.")
os.environ["YOLO_VERBOSE"] = "false"  # Hilangkan progress bar repetitif; metrics tetap dihitung tiap epoch.
os.environ["YOLO_AUTOINSTALL"] = "false"
WORKDIR = Path("/kaggle/working")
REPO_URL = "https://github.com/danial2015/yolo-aceh-rdd2022.git"
BRANCH = "feat/yolo12s-lghost-ema32"
REPO_DIR = WORKDIR / "yolo12s-lghost-ema32"


def run(command, cwd=None):
    subprocess.run(command, cwd=cwd, check=True)


def git_output(*args):
    return subprocess.check_output(["git", *args], cwd=REPO_DIR, text=True).strip()


if not REPO_DIR.exists():
    run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(REPO_DIR)])
else:
    assert git_output("remote", "get-url", "origin") == REPO_URL, "Folder berisi repo berbeda."
    assert not git_output("status", "--porcelain"), "Repo memiliki perubahan lokal; gunakan folder baru."
    assert git_output("branch", "--show-current") == BRANCH, "Branch tidak cocok; gunakan folder baru."
    run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)

REPO_COMMIT = git_output("rev-parse", "HEAD")
run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(REPO_DIR)])
sys.path.insert(0, str(REPO_DIR))
# Imports wajib dilakukan SETELAH repo di-install.
import torch  # noqa: E402
import ultralytics  # noqa: E402
from ultralytics import YOLO  # noqa: E402
from ultralytics.models.yolo.detect.lghost import LGhostDetectionTrainer  # noqa: E402

assert Path(ultralytics.__file__).resolve().is_relative_to(REPO_DIR.resolve())
assert torch.cuda.is_available(), "Aktifkan GPU Kaggle."
print(f"Source: {BRANCH} @ {REPO_COMMIT}")
print(f"PyTorch: {torch.__version__} | GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Konfigurasi: hyperparameter umum mengikuti notebook EMA-Ghost sebelumnya.
import json
from datetime import datetime
from ultralytics.utils import YAML

DATA_ROOT = Path("/kaggle/input/datasets/danialalfayyadh/ch-rdd-2022/datasets-china-split-fix")
# Jika input Kaggle Anda memakai path lama, ubah DATA_ROOT menjadi:
# /kaggle/input/datasets/danialalfayyadh/ch-rdd2022/datasets-china-split
DATASET_NAME = "china"  # Ganti "local" untuk eksperimen dataset lokal TERPISAH.
NAMES = {0: "D00", 1: "D10", 2: "D20", 3: "D40", 4: "Repair"}
EPOCHS, IMGSZ, BATCH, NBS = 160, 640, 16, 64
OPTIMIZER, LR0, MOMENTUM, WEIGHT_DECAY = "SGD", 0.01, 0.937, 0.0005
SEED, WORKERS, LOG_EVERY = 42, 2, 20
DEVICE = 0
MODEL_YAML = REPO_DIR / "ultralytics/cfg/models/12/yolo12s-lghost-ema32.yaml"
EXPERIMENT = f"yolo12s_lghost_ema32_{DATASET_NAME}_{datetime.now():%Y%m%d_%H%M%S}"
ARTIFACT_DIR = WORKDIR / EXPERIMENT
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
DATA_YAML = ARTIFACT_DIR / "dataset.yaml"

for split in ("train", "val", "test"):
    image_dir = DATA_ROOT / split / "images"
    label_dir = DATA_ROOT / split / "labels"
    assert image_dir.is_dir() and label_dir.is_dir(), f"Split/labels tidak ditemukan: {split}"
    count = sum(p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"} for p in image_dir.rglob("*"))
    assert count > 0 and any(label_dir.rglob("*.txt")), f"Split kosong atau tidak berlabel: {split}"
    print(f"{split}: {count:,} images")
YAML.save(
    DATA_YAML,
    dict(path=str(DATA_ROOT), train="train/images", val="val/images", test="test/images", nc=len(NAMES), names=NAMES),
)
pruning = YAML.load(MODEL_YAML)["lghost_pruning"]
print(f"LG groups=4 | RA={pruning['ratio']} | prune setelah epoch 32, 64, 96 untuk epochs=160")
MANIFEST = dict(
    repository=REPO_URL,
    branch=BRANCH,
    commit=REPO_COMMIT,
    dataset=DATASET_NAME,
    dataset_root=str(DATA_ROOT),
    model_yaml=str(MODEL_YAML),
    pruning=pruning,
    torch=torch.__version__,
    ultralytics=ultralytics.__version__,
)
(ARTIFACT_DIR / "manifest.json").write_text(json.dumps(MANIFEST, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "requirements-frozen.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8"
)

In [ ]:
# 3. Print model dan audit pretrained asli sebelum training.
from ultralytics.nn.modules.lghost import LGhostBottleneck, LearnedGroupConv
from ultralytics.nn.tasks import DetectionModel

source = YOLO("yolo12s.pt")  # Download checkpoint resmi; bukan model dari YAML kosong.
inspection = DetectionModel(str(MODEL_YAML), nc=len(NAMES), verbose=False)
report = inspection.load(source.model, verbose=False)
assert report["mode"] == "semantic"
assert report["exact_transferred_parameter_elements"] > 8_000_000
assert [i for i, m in enumerate(inspection.model) if isinstance(m, LGhostBottleneck)] == [16, 20]
print(json.dumps(report, indent=2))
print(inspection)
print(f"Target parameters: {sum(p.numel() for p in inspection.parameters()):,}")
(ARTIFACT_DIR / "model.txt").write_text(str(inspection), encoding="utf-8")
(ARTIFACT_DIR / "preflight_transfer.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
# Audit training di cell berikut memastikan trainer benar-benar memuat bobot yang sama ke target nc=5.
del inspection, source
torch.cuda.empty_cache()
model = YOLO(str(MODEL_YAML), task="detect")

In [ ]:
# 4. Training: detail ke training.log, ringkasan ke layar tiap 20 epoch.
import contextlib
import logging
from ultralytics.utils import LOGGER

TRAIN_LOG = ARTIFACT_DIR / "training.log"
console = sys.stdout
reported_epochs = set()


def audit_training_transfer(trainer):
    report = trainer.model.pretrained_transfer_report
    assert report["mode"] == "semantic", "Pretrained tidak memakai semantic mapping."
    assert report["exact_transferred_parameter_elements"] > 8_000_000
    print(
        f"PRETRAINED OK: {report['exact_transferred_tensors']} tensors; "
        f"{report['exact_transferred_parameter_elements']:,} parameter elements",
        file=console,
        flush=True,
    )


def print_epoch_summary(trainer):
    epoch = trainer.epoch + 1
    if epoch in reported_epochs or (epoch != 1 and epoch % LOG_EVERY and epoch != trainer.epochs):
        return
    reported_epochs.add(epoch)
    values = {k: round(float(v), 6) for k, v in trainer.metrics.items()}
    stages = [int(m.stage) for m in trainer.model.modules() if isinstance(m, LearnedGroupConv)]
    print(f"Epoch {epoch}/{trainer.epochs} | LG stages={stages} | {values}", file=console, flush=True)


model.add_callback("on_pretrain_routine_end", audit_training_transfer)
model.add_callback("on_fit_epoch_end", print_epoch_summary)
print(f"Training dimulai. Log lengkap: {TRAIN_LOG}")
handlers, level = LOGGER.handlers[:], LOGGER.level
try:
    with TRAIN_LOG.open("w", encoding="utf-8") as stream:
        LOGGER.handlers = [logging.StreamHandler(stream)]
        LOGGER.setLevel(logging.INFO)
        with contextlib.redirect_stdout(stream), contextlib.redirect_stderr(stream):
            model.train(
                trainer=LGhostDetectionTrainer,
                data=str(DATA_YAML),
                epochs=EPOCHS,
                imgsz=IMGSZ,
                batch=BATCH,
                nbs=NBS,
                device=DEVICE,
                workers=WORKERS,
                project=str(ARTIFACT_DIR),
                name="train",
                exist_ok=False,
                pretrained="yolo12s.pt",
                optimizer=OPTIMIZER,
                lr0=LR0,
                momentum=MOMENTUM,
                weight_decay=WEIGHT_DECAY,
                cos_lr=False,
                patience=0,
                seed=SEED,
                compile=False,
                time=None,
                plots=True,
                verbose=False,
            )
finally:
    LOGGER.handlers, LOGGER.level = handlers, level
    if model.trainer is not None:
        RUN_DIR = Path(model.trainer.save_dir)
        print(f"Checkpoint/log training: {RUN_DIR}")
BEST_PT, LAST_PT = Path(model.trainer.best), Path(model.trainer.last)
assert BEST_PT.exists() and LAST_PT.exists()
print(f"Training selesai. Best: {BEST_PT}")

In [ ]:
# 5. Validation -> test. Test tidak digunakan untuk memilih hyperparameter/checkpoint.
best_model = YOLO(str(BEST_PT))
checkpoint_stages = [int(m.stage) for m in best_model.model.modules() if isinstance(m, LearnedGroupConv)]
print(f"Pruning stage best.pt: {checkpoint_stages} (final=3). Best bisa berasal dari sebelum pruning selesai.")


def summarize(metrics):
    return dict(
        precision=float(metrics.box.mp),
        recall=float(metrics.box.mr),
        map50=float(metrics.box.map50),
        map50_95=float(metrics.box.map),
        per_class_map50_95=[float(x) for x in metrics.box.maps],
    )


evaluation = {"checkpoint": str(BEST_PT), "lg_stages": checkpoint_stages}
for split in ("val", "test"):
    metrics = best_model.val(
        data=str(DATA_YAML),
        split=split,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        project=str(ARTIFACT_DIR),
        name=split,
        exist_ok=False,
        plots=True,
        verbose=False,
    )
    evaluation[split] = summarize(metrics)
    print(f"{split}: {evaluation[split]}")
(ARTIFACT_DIR / "evaluation.json").write_text(json.dumps(evaluation, indent=2), encoding="utf-8")

In [ ]:
# 6. Opsional: compact BEST yang sudah fully-pruned, tanpa mengubah bobot hasil training.
# Jangan memaksa pruning pada best.pt yang masih tahap awal: itu mengubah model tanpa fine-tuning.
from ultralytics.nn.modules.lghost import compact_lghost
from copy import deepcopy

if all(stage == 3 for stage in checkpoint_stages):
    compact = deepcopy(best_model)
    compact.model = compact_lghost(best_model.model).float().cpu().eval()
    original = deepcopy(best_model.model).float().cpu().eval()
    with torch.inference_mode():
        sample = torch.rand(1, 3, 128, 128)
        expected = original(sample)[0]
        actual = compact.model(sample)[0]
        torch.testing.assert_close(actual, expected, rtol=1e-4, atol=1e-4)
    compact.save(ARTIFACT_DIR / "best_compact_inference.pt")
    print("Compact checkpoint tersimpan (inference-only; bukan untuk resume).")
    del compact, original
else:
    print("Compact best dilewati: best.pt belum fully-pruned. last.pt tetap tersedia.")
# best.pt standar tidak diganti; metrics di atas tetap untuk best.pt asli.

In [ ]:
# 7. ZIP hasil, metadata commit, YAML, dokumentasi, dan notebook.
# Cell ini juga dapat dijalankan setelah training terinterupsi jika ARTIFACT_DIR masih tersedia.
import shutil
import zipfile
from IPython.display import FileLink, display

shutil.copy2(MODEL_YAML, ARTIFACT_DIR / MODEL_YAML.name)
shutil.copy2(REPO_DIR / "README_LGHOST_EMA32.md", ARTIFACT_DIR / "README_LGHOST_EMA32.md")
shutil.copy2(REPO_DIR / "examples/kaggle_yolo12s_lghost_ema32_rdd2022.ipynb", ARTIFACT_DIR / "source_notebook.ipynb")
ZIP_PATH = ARTIFACT_DIR.with_suffix(".zip")
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in ARTIFACT_DIR.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(ARTIFACT_DIR.parent))
print(f"ZIP: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1024**2:.1f} MB)")
display(FileLink(str(ZIP_PATH)))
print("Simpan/download output sebelum mematikan session Kaggle. ZIP bukan backup otomatis lintas session.")

## Melanjutkan training yang terputus
Pastikan repo/commit dan dataset sama. Gunakan **last.pt yang masih menyimpan optimizer**, bukan compact checkpoint:
```python
continued = YOLO("/kaggle/input/your-checkpoint/last.pt")
continued.train(trainer=LGhostDetectionTrainer, resume=True, device=0)
```
Checkpoint dari training yang sudah selesai biasanya sudah di-strip optimizer dan bukan checkpoint resume.
Jangan mengubah total epoch atau RA saat resume: jadwal dan mask pruning harus konsisten.
Semua metrics per epoch tetap tersimpan di `train/results.csv`; interval 20 hanya mengatur tampilan.
`EMA-32` attention berbeda dengan `ModelEMA` rata-rata bobot bawaan Ultralytics.
